# 7. Hafta — Modeli Doğru Değerlendirmek

**BTP271 Uygulamalı Yapay Zeka** · Ön-Yüz Yazılım Geliştirme
Öğr. Gör. Çisem YAŞAR · cyasar@gelisim.edu.tr

---
## 0. Yeni veri setimiz: Meme kanseri (Wisconsin)

569 hastadan alınan doku örneklerinin görüntülerinden ölçülmüş **30 öznitelik** (hücre çekirdeğinin yarıçapı, alanı, pürüzsüzlüğü…) ve her örneğin **iyi huylu / kötü huylu** olduğu bilgisi.
Scikit-learn'ün içinde hazır gelir, internetten indirmeye gerek yoktur.

> ⚕️ **Not:** Bu, eğitim amaçlı kullanılan anonim bir veri setidir. Kuracağımız model bir **teşhis aracı değildir**; ama yapacağı her hatanın gerçek hayatta bir insanı etkileyeceğini unutmadan çalışacağız.

### Etiketi çevirmek
Veri setinde `target` sütunu **0 = kötü huylu, 1 = iyi huylu** şeklinde kodlanmış. Precision ve recall gibi metrikler **1'i "pozitif", yani aradığımız durum** kabul eder.
Biz kötü huylu tümörü **yakalamak** istediğimiz için etiketi çeviriyoruz: **1 = kötü huylu.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

veri = load_breast_cancer(as_frame=True)
df = veri.frame
df["kotu_huylu"] = 1 - df["target"]        # 0 ↔ 1 çevir: artık 1 = kötü huylu

X = df[veri.feature_names]                 # 30 öznitelik
y = df["kotu_huylu"]

print(df.shape)
df[["mean radius", "mean area", "mean smoothness", "kotu_huylu"]].head()

#### ✍️ Kendin Yaz — Veriyi tanı

1. Kaç hasta ve kaç öznitelik var?
2. Kötü huylu örneklerin **oranını** `kotu_orani` değişkenine ata.
3. `X` içinde hiç eksik değer var mı? Toplam eksik değer sayısını `eksik_sayisi` değişkenine ata.
4. `"mean area"` ve `"mean smoothness"` sütunlarının en küçük ve en büyük değerlerini yazdır.

Yorum satırına cevapla: 4. adımdaki iki sütunun değer aralıkları arasında nasıl bir fark var?

In [ ]:
# kodunu yaz


# Cevabım:

In [ ]:
# 🧪 Test hücresi — değiştirme, sadece çalıştır
assert X.shape == (569, 30)
assert abs(kotu_orani - 212 / 569) < 1e-9
assert eksik_sayisi == 0
print("✅ Tüm testler geçti!")

---
## 1. Eğitim ve test verisi

6\. haftada modelleri **eğitildikleri verinin aynısında** ölçtük. Bu, bir öğrenciyi çalıştığı soruların aynısıyla sınava sokmaya benzer: yüksek not, gerçekten öğrendiğini göstermez, **ezberlemiş** de olabilir.

Çözüm: Veriyi baştan ikiye ayırmak.
- **Eğitim verisi (train):** Model sadece bundan öğrenir.
- **Test verisi (test):** Model bunu **hiç görmez**; sadece en sonda başarıyı ölçmek için kullanılır.

| Parametre | Ne işe yarar? |
|---|---|
| `test_size=0.2` | Verinin %20'si teste, %80'i eğitime ayrılır |
| `random_state=42` | Ayırma rastgeledir; bu sayı sayesinde herkes **aynı** ayrımı alır
| `stratify=y` | Kötü huylu oranı eğitimde ve testte **aynı** kalır |

In [ ]:
from sklearn.model_selection import train_test_split

X_egitim, X_test, y_egitim, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Eğitim:", X_egitim.shape, "| Test:", X_test.shape)
print(f"Kötü huylu oranı → eğitim: %{y_egitim.mean() * 100:.1f} | test: %{y_test.mean() * 100:.1f}")

#### ✍️ Kendin Yaz — İki ayrı sınav
1. `LogisticRegression(max_iter=5000)` modelini **sadece eğitim verisiyle** eğit, `model` değişkenine ata.

2. Modelin **eğitim** verisindeki doğruluğunu `dogruluk_egitim`, **test** verisindeki doğruluğunu `dogruluk_test` değişkenine ata.

3. İkisini yüzde olarak yazdır.

Yorum satırına cevapla: Hangisi daha yüksek? Hangisi modelin **gerçek** başarısını gösterir, neden?

In [ ]:
# kodunu yaz


# Cevabım:

In [ ]:
# 🧪 Test hücresi
assert 0.90 < dogruluk_test < 0.98
assert dogruluk_egitim >= dogruluk_test
print("✅ Tüm testler geçti!")

---
## 2. Doğruluk neden yetmez?

Hiçbir şey öğrenmeyen, **herkese "iyi huylu" diyen** bir model düşünelim:

In [ ]:
hep_iyi_huylu = np.zeros(len(y_test))          # herkes için 0 (iyi huylu) tahmini
print(f"Hiç öğrenmeyen modelin doğruluğu: %{accuracy_score(y_test, hep_iyi_huylu) * 100:.1f}")

%63! Çünkü hastaların çoğu zaten iyi huylu. Ama bu model **kötü huylu tümörlerin hiçbirini yakalayamıyor.** Doğruluk, hangi tür hatayı yaptığımızı söylemiyor.

### Karmaşıklık matrisi (confusion matrix)
Modelin tahminlerini gerçek değerlerle dört kutuda karşılaştırır:

| | Tahmin: iyi huylu (0) | Tahmin: kötü huylu (1) |
|---|---|---|
| **Gerçek: iyi huylu (0)** | ✅ Doğru negatif (**TN**) | ⚠️ Yanlış alarm (**FP**) |
| **Gerçek: kötü huylu (1)** | ❌ Kaçırılan tümör (**FN**) | ✅ Doğru pozitif (**TP**) |

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

tahmin = model.predict(X_test)
matris = confusion_matrix(y_test, tahmin)
print(matris)

ConfusionMatrixDisplay(matris, display_labels=["iyi huylu", "kötü huylu"]).plot(cmap="Blues")
plt.title("Karmaşıklık matrisi (test verisi)")
plt.xlabel("Tahmin")
plt.ylabel("Gerçek")
plt.show()

### Precision, recall ve F1
| Metrik | Formül | Sorduğu soru |
|---|---|---|
| **Precision (kesinlik)** | TP / (TP + FP) | "Kötü huylu" dediklerimin kaçı gerçekten kötü huylu? |
| **Recall (duyarlılık)** | TP / (TP + FN) | Gerçek kötü huylu tümörlerin kaçını yakaladım? |
| **F1** | 2 × P × R / (P + R) | Precision ve recall'un dengeli ortalaması |

#### ✍️ Kendin Yaz — Metrikleri kendin hesapla
1. `matris` içinden dört değeri al: `tn, fp, fn, tp = matris.ravel()`
2. Precision, recall ve F1'i **formüllerle** hesapla; `precision`, `recall`, `f1` değişkenlerine ata.
3. Aynı değerleri `precision_score`, `recall_score`, `f1_score` fonksiyonlarıyla hesaplayıp yan yana yazdır; aynı çıktığını gör.

Yorum satırına cevapla: Model kaç kötü huylu tümörü **kaçırdı**? Bu, hangi metriği düşürüyor?

In [ ]:
# kodunu yaz


# Cevabım:

In [ ]:
# 🧪 Test hücresi
assert abs(precision - precision_score(y_test, tahmin)) < 1e-9
assert abs(recall - recall_score(y_test, tahmin)) < 1e-9
assert abs(f1 - f1_score(y_test, tahmin)) < 1e-9
print("✅ Tüm testler geçti!")

---
## 3. Precision mı, recall mı? Karar eşiğini değiştirmek

Kanser taramasında **bir tümörü kaçırmak (FN)**, sağlıklı birine **yanlış alarm vermekten (FP)** çok daha tehlikelidir: yanlış alarm ek bir testle düzeltilir, kaçırılan tümör düzeltilemeyebilir. Bu yüzden burada **recall** daha önemlidir.

6\. haftadan hatırla: Lojistik regresyon bir **olasılık** verir ve varsayılan olarak **0.5**'in üstünü pozitif sayar. Bu eşiği düşürürsek model daha kolay "kötü huylu" der.

#### ✍️ Kendin Yaz — Eşiği 0.3'e indir
1. Test verisi için kötü huylu olasılıklarını `olasiliklar` değişkenine ata
2. Olasılığı **0.3 ve üzeri** olanları kötü huylu say
3. Bu tahminlerin karmaşıklık matrisini, precision ve recall değerlerini hesapla
4. 0.5 eşiğiyle yan yana karşılaştıran bir tablo yazdır.

Yorum satırına cevapla: Kaçırılan tümör ve yanlış alarm sayıları nasıl değişti? Bir kanser tarama uygulaması için hangi eşiği seçerdin?

In [ ]:
# kodunu yaz


# Cevabım:

In [ ]:
# 🧪 Test hücresi
assert recall_03 >= recall_score(y_test, tahmin)
assert precision_03 <= precision_score(y_test, tahmin)
assert matris_03.sum() == len(y_test)
print("✅ Tüm testler geçti!")

### Tek satırda rapor: `classification_report`
Tüm metrikleri her sınıf için bir arada gösterir:

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, tahmin, target_names=["iyi huylu", "kötü huylu"]))

---
## 🏁 Haftanın Mini Projesi


Bir doktor, 30 ölçümün hepsi yerine sadece şu **5 ölçümle** çalışan daha basit bir model istiyor:
`["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]`

**Görevler:**
1. Bu 5 sütunla `X5` tablosunu oluştur ve **aynı parametrelerle** (`test_size=0.2, random_state=42, stratify=y`) eğitim / test olarak ayır.
2. `LogisticRegression(max_iter=5000)` ile `model5`'i eğit ve test verisinde tahmin yap.
3. Karmaşıklık matrisini çizdir ve `cm5.png` olarak kaydet.
4. İki modeli karşılaştıran bir tablo oluştur ve `tablo` değişkenine ata: satırlar **"30 öznitelik"** ve **"5 öznitelik"**; sütunlar **Accuracy, Precision, Recall, F1** (test verisinde, 3 basamak).
5. Yorum satırına cevapla: Basit model hangi metrikte en çok kaybetti? Doktora hangisini önerirsin?

In [ ]:
# kodunu yaz


# Cevabım:

In [ ]:
# 🧪 Test hücresi
import os
assert list(tablo.columns) == ["Accuracy", "Precision", "Recall", "F1"]
assert list(tablo.index) == ["30 öznitelik", "5 öznitelik"]
assert tablo.loc["30 öznitelik", "F1"] > tablo.loc["5 öznitelik", "F1"]
assert os.path.exists("cm5.png")
print("✅ Tüm testler geçti! Tebrikler 🎉")